### Imports

In [1]:
import json
import time
from pathlib import Path

import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader

from src.config import CONFIG
from src.lstm_utils.lstm_dataset import LSTMDataset
from src.lstm_utils.lstm_tokeniser import LSTMTokeniser
from src.lstm_utils.lstm_training import train_one_epoch, validate, evaluate
from src.lstm_utils.lstm_tuning import run_hyperparameter_sweep
from src.models.lstm_classifier import LSTMClassifier

/Users/jackrong/University/34812-cwk-S-Project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Constants

In [2]:
TRAIN_DATA_PATH = Path(f'../data/train.csv')
VAL_DATA_PATH = Path(f'../data/dev.csv')
BENCHMARK_DATA_PATH = Path(f'../data/NLI_trial.csv')
TEST_DATA_PATH = Path(f'../data/test.csv')

HYPERPARAMETERS_PATH = Path(f'../hyperparameters/lstm.json')
MODEL_PATH = Path(f'../models/lstm.pt')

### Device

In [3]:
device = torch.device(
    'cuda' if torch.cuda.is_available() else
    'mps' if torch.backends.mps.is_available() else
    'cpu'
)
print(f'Using device: {device}')

Using device: mps


### Set a constant seed for reproducability

In [4]:
generator = torch.manual_seed(CONFIG.seed)

### Verify tokenising works

In [5]:
def verify_tokeniser() -> None:
    train_df = pd.read_csv(TRAIN_DATA_PATH)
    sample = train_df.sample(1).iloc[0]
    sample_premise = sample['premise']

    lstm_tokeniser = LSTMTokeniser()
    premise_ids, premise_mask = lstm_tokeniser.encode(sample_premise, CONFIG.lstm.max_length)
    print(f'Sample premise: {sample_premise}')
    print(f'Premise ids: {premise_ids}')
    print(f'Premise mask: {premise_mask}')
    assert(len(premise_ids) == CONFIG.lstm.max_length)

# verify_tokeniser()

### Set up Datasets and DataLoaders

In [6]:
lstm_tokeniser = LSTMTokeniser()

train_dataset = LSTMDataset(TRAIN_DATA_PATH, lstm_tokeniser)
val_dataset = LSTMDataset(VAL_DATA_PATH, lstm_tokeniser)
test_dataset = LSTMDataset(BENCHMARK_DATA_PATH, lstm_tokeniser)

train_dataloader = DataLoader(train_dataset, generator=generator, batch_size=CONFIG.batch_size, shuffle=True, drop_last=True)
val_dataloader = DataLoader(val_dataset, generator=generator, batch_size=CONFIG.batch_size, shuffle=False)
test_dataloader = DataLoader(test_dataset, generator=generator, batch_size=CONFIG.batch_size, shuffle=False)

### Hyperparameter Tuning

In [7]:
if CONFIG.lstm.hyperparameter_tuning.should_run:
    run_hyperparameter_sweep(device, lstm_tokeniser, train_dataloader, val_dataloader, HYPERPARAMETERS_PATH)

### Training Loop

In [8]:
if __name__ == '__main__':
    hyperparameters = json.load(open(HYPERPARAMETERS_PATH, 'r'))
    print(f'Hyperparameters used: {hyperparameters}')
    print()

    model = LSTMClassifier(lstm_tokeniser).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=hyperparameters['lr'])
    criterion = nn.CrossEntropyLoss()

    best_acc = 0.0
    patience = 0
    train_losses, train_accs = [], []
    val_losses, val_accs = [], []
    total_start = time.time()
    for epoch in range(CONFIG.epochs):
        epoch_start = time.time()
        print(f'[Epoch {epoch + 1}/{CONFIG.epochs}]')

        train_loss, train_acc = train_one_epoch(device, model, criterion, optimizer, train_dataloader)
        train_losses.append(train_loss)
        train_accs.append(train_acc)

        val_loss, val_acc = validate(device, model, criterion, val_dataloader)
        val_losses.append(val_loss)
        val_accs.append(val_acc)

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), MODEL_PATH)
            patience = 0
        else:
            patience += 1

        if patience == CONFIG.patience:
            print(f'Finishing training early, no improvement in {CONFIG.patience} epochs')
            break

        epoch_time = time.time() - epoch_start
        elapsed = time.time() - total_start
        print(f'Train Loss: {train_loss:.2f} | Train Accuracy: {train_acc:.2f}% | Val Loss: {val_loss:.2f} | Val Accuracy: {val_acc:.2f}%')
        print(f'Epoch time: {epoch_time:.1f}s | Total elapsed: {elapsed:.1f}s')
        print()

    total_time = time.time() - total_start
    print(f'Training complete in {total_time:.1f}s')
    print(f'Best model had an accuracy of {best_acc:.2f}%.')

    print('Running final evaluation on NLI_trial.csv...')
    model.load_state_dict(torch.load(MODEL_PATH))
    test_results = evaluate(device, model, test_dataloader)
    print(f'NLI_trial — Accuracy: {test_results["accuracy"]:.2f}% | F1 (weighted): {test_results["f1_weighted"]:.4f}')

Hyperparameters used: {'lr': 0.0015903715140364574}

[Epoch 1/100]
Train Loss: 7.54 | Train Accuracy: 61.44% | Val Loss: 0.60 | Val Accuracy: 66.79%
Epoch time: 70.0s | Total elapsed: 70.0s

[Epoch 2/100]
Train Loss: 10.84 | Train Accuracy: 67.65% | Val Loss: 0.58 | Val Accuracy: 67.64%
Epoch time: 66.1s | Total elapsed: 136.1s

[Epoch 3/100]
Train Loss: 8.33 | Train Accuracy: 71.85% | Val Loss: 0.59 | Val Accuracy: 67.44%
Epoch time: 64.2s | Total elapsed: 200.3s

[Epoch 4/100]
Train Loss: 12.42 | Train Accuracy: 76.11% | Val Loss: 0.61 | Val Accuracy: 67.04%
Epoch time: 73.3s | Total elapsed: 273.5s

[Epoch 5/100]


KeyboardInterrupt: 